# Windowed CNN blind decoder - Google-Colab GPU training (DIV2K)

PyTorch pipeline that trains the experimental **windowed 1D-CNN** blind decoder on DIV2K.
Every training sample is generated by the **actual, frozen project embedder**
(`src/watermark/embed.py`) with payloads from the real registry codec
(`src/watermark/id_registry.py`), so the network learns the exact signal the production
system embeds.

**Honesty box - read first:**

- **No faked results.** Every number (BER, exact registry-ID recovery, false-positive
  rate) is computed on the *test* split and written to `metrics.json`.
- **Split is 700 train / 100 validation / 100 test.** Released DIV2K has only 900 images
  (800 train + 100 valid); "800/100/100" needs 1000. Train = 0001-0700, validation =
  0701-0800 (official-train tail), test = 0801-0900 (the Official VALID set - the same
  test the local project uses).
- The windowed decoder does **not** replace the shipped decoder by default; artifacts are
  brought back and used through an opt-in `DECODER_MODE=windowed_cnn`.
- This notebook is a thin driver for `training/colab_train_decoder.py` (the actual
  implementation lives there - read its docstring for every decision).

Run the cells in order; the pipeline is resumable - caches and checkpoints survive a
runtime disconnect, and Stage C auto-resumes from `windowed_cnn_best.pt`.
**Getting the project sources into Colab (Cell 2):**

1. **Google Drive (recommended):** upload `colab_payload.zip` ONCE into the root of
   your Drive (build it on your machine with `python scripts/build_colab_payload.py`).
   Cell 2 mounts Drive and unzips from there - Drive survives disconnects, so after
   every reconnect you only click the one-time "Connect to Google Drive" popup. (leave
   Cell 2 defaults as-is for this mode)
2. **Manual upload (no Drive):** Cell 2 opens a file picker - choose `colab_payload.zip`
   there. On a disconnect you must re-upload it.


In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile

print("Python", sys.version.split()[0])
for pkg in ["pywavelets", "opencv-python-headless", "kagglehub", "pyyaml", "numpy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU - will train on CPU)"
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| gpu:", gpu)

In [ ]:
import os  # noqa: F811 (by design: this cell runs standalone on a fresh kernel)
import shutil  # noqa: F811
import sys
import zipfile  # noqa: F811

PROJECT_ROOT = "/content/deep-watermarking"
EXISTS = os.path.isdir(PROJECT_ROOT) and os.path.isfile(
    os.path.join(PROJECT_ROOT, "COLLAB_README.md")
)
if not EXISTS and os.path.isdir(PROJECT_ROOT):
    print("rebuilding unverified project dir (missing canary)")
    shutil.rmtree(PROJECT_ROOT)

ZIP_NAME = "colab_payload.zip"      # create it with:  python scripts/build_colab_payload.py
DRIVE_DIR = "/content/drive/MyDrive"  # where that zip lives on Google Drive

def _unzip_to_disk(zip_path: str) -> None:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(PROJECT_ROOT)
    assert os.path.isfile(os.path.join(PROJECT_ROOT, "COLLAB_README.md")), "bad payload: no canary"
    print("unzipped project sources into", PROJECT_ROOT)

if not os.path.isdir(PROJECT_ROOT):
    zip_path = ""
    try:
        from google.colab import drive

        drive.mount("/content/drive")  # click "Connect to Google Drive" once per session
        candidate = os.path.join(DRIVE_DIR, ZIP_NAME)
        if os.path.isfile(candidate):
            zip_path = candidate
            print("sources found on Google Drive:", zip_path)
    except Exception as exc:  # noqa: BLE001 - fall back to the file picker
        print("Google Drive unavailable, using the file picker:", exc)
    if not zip_path:
        from google.colab import files

        print("Upload the zip (colab_payload.zip) in the file chooser that appears now")
        uploaded = files.upload()
        name, data = next(iter(uploaded.items()))
        zip_path = os.path.join(os.path.dirname(PROJECT_ROOT), name)
        with open(zip_path, "wb") as fh:
            fh.write(data)
    _unzip_to_disk(zip_path)
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "training"))
import colab_train_decoder as ctd  # the actual pipeline

print(
    "env:",
    {
        k: v
        for k, v in ctd.check_environment().items()
        if k in ("torch", "cuda_available", "gpu_name", "numpy")
    },
)


In [ ]:
# resume-safe: kernel restarts clear globals - re-import the pipeline module + config
if "ctd" not in globals():
    import importlib.util
    import sys
    if "PROJECT_ROOT" not in globals():
        PROJECT_ROOT = "/content/deep-watermarking"
    sys.path.insert(0, PROJECT_ROOT)
    if not sys.modules.get("ctd_mod"):
        spec = importlib.util.spec_from_file_location(
            "ctd_mod", f"{PROJECT_ROOT}/training/colab_train_decoder.py"
        )
        mod = importlib.util.module_from_spec(spec)
        sys.modules["ctd_mod"] = mod
        spec.loader.exec_module(mod)
    ctd = sys.modules["ctd_mod"]
if "CFG" not in globals():
    CFG = ctd.load_pipeline_config(PROJECT_ROOT, {}).resolved()

# Pipeline configuration (overridable; read from configs/windowed_cnn.yaml + CLI defaults)
CFG = ctd.load_pipeline_config(PROJECT_ROOT, {}).resolved()
print("bit_length", CFG.bit_length, "| window", CFG.window_size, "| image_size", CFG.image_size)
print(
    "alpha",
    CFG.alpha,
    "| epochs",
    CFG.epochs,
    "| batch",
    CFG.batch_size,
    "| lr",
    CFG.lr,
    "| patience",
    CFG.patience,
)
print("split_mode", CFG.split_mode, "| config hash", CFG.config_hash())

### Stage 0 - prove the actual embedder runs here
A synthetic 256x256 image goes through the frozen `src.watermark.embed.embed` and the
authoritative per-bit window feature path. If this cell fails, STOP - the training signal
would not be authentic.

In [ ]:
# resume-safe: kernel restarts clear globals - re-import the pipeline module + config
if "ctd" not in globals():
    import importlib.util
    import sys
    if "PROJECT_ROOT" not in globals():
        PROJECT_ROOT = "/content/deep-watermarking"
    sys.path.insert(0, PROJECT_ROOT)
    if not sys.modules.get("ctd_mod"):
        spec = importlib.util.spec_from_file_location(
            "ctd_mod", f"{PROJECT_ROOT}/training/colab_train_decoder.py"
        )
        mod = importlib.util.module_from_spec(spec)
        sys.modules["ctd_mod"] = mod
        spec.loader.exec_module(mod)
    ctd = sys.modules["ctd_mod"]
if "CFG" not in globals():
    CFG = ctd.load_pipeline_config(PROJECT_ROOT, {}).resolved()

import numpy as np

rng_img = np.random.default_rng(7)
rgb = (rng_img.random((256, 256, 3)) * 255).astype(np.uint8)
bits = [0, 1] * 32
wm = ctd.embed(rgb, bits, ctd.EmbedConfig(alpha=CFG.alpha, bit_length=CFG.bit_length))
assert wm.watermarked_image.shape == rgb.shape
se = np.mean((wm.watermarked_image.astype(np.float64) - rgb.astype(np.float64)) ** 2)
print(f"embedder sanity: PSNR {10 * np.log10(255**2 / max(se, 1e-12)):.2f} dB (alpha={CFG.alpha})")
w = ctd.synthesize_windows(wm.watermarked_image, bits, CFG)
assert w.shape == (CFG.bit_length, CFG.window_size) and np.isfinite(w).all()
print("window feature path OK:", w.shape)

### Stage A - data: DIV2K high-resolution images
Primary source is KaggleHub (`soumikrakshit/div2k-high-resolution-images`, no API key
needed). If it fails on this runtime, set `SOURCE = "official"` to pull the official ETHz
archives with wget; or `"fake"` to run a **dev-only** wiring test (trains on noise -
numbers meaningless).

In [ ]:
# resume-safe: kernel restarts clear globals - re-import the pipeline module + config
if "ctd" not in globals():
    import importlib.util
    import sys
    if "PROJECT_ROOT" not in globals():
        PROJECT_ROOT = "/content/deep-watermarking"
    sys.path.insert(0, PROJECT_ROOT)
    if not sys.modules.get("ctd_mod"):
        spec = importlib.util.spec_from_file_location(
            "ctd_mod", f"{PROJECT_ROOT}/training/colab_train_decoder.py"
        )
        mod = importlib.util.module_from_spec(spec)
        sys.modules["ctd_mod"] = mod
        spec.loader.exec_module(mod)
    ctd = sys.modules["ctd_mod"]
if "CFG" not in globals():
    CFG = ctd.load_pipeline_config(PROJECT_ROOT, {}).resolved()

SOURCE = "kagglehub"  # "kagglehub" | "official" | "fake"
import shutil
import urllib.request
from pathlib import Path

train_files = valid_files = None
if SOURCE == "fake":
    train_files, valid_files = ctd.download_div2k(CFG, use_kagglehub=False, fake=True)
elif SOURCE == "kagglehub":
    try:
        train_files, valid_files = ctd.download_div2k(CFG, use_kagglehub=True)
    except ctd.ColabPipelineError as exc:
        print("KaggleHub failed - switching to official archives.\nreason:", exc)
        SOURCE = "official"
if SOURCE == "official":
    base = "https://data.vision.ee.ethz.ch/cvl/DIV2K/"
    raw = Path(PROJECT_ROOT) / "data" / "raw"
    raw.mkdir(parents=True, exist_ok=True)
    for part in ["DIV2K_train_HR.zip", "DIV2K_valid_HR.zip"]:
        dest = raw / part
        if not dest.is_file():
            print("downloading", part)
            urllib.request.urlretrieve(base + part, dest)
    for part in ["DIV2K_train_HR", "DIV2K_valid_HR"]:
        z = raw / f"{part}.zip"
        if z.is_file():
            with zipfile.ZipFile(z) as zf:
                zf.extractall(raw)
            z.unlink(missing_ok=True)
    # official archives nest once:  <raw>/<part>/<part>/0001.png  ->  flatten
    for part in ["DIV2K_train_HR", "DIV2K_valid_HR"]:
        outer = raw / part
        inner = outer / part
        if inner.is_dir() and not any(outer.glob("*.png")):
            for f in inner.iterdir():
                shutil.move(str(f), outer)
            inner.rmdir()
    train_files, valid_files = ctd.discover_div2k_pngs(raw)
print(f"DIV2K ready: train={len(train_files)} valid={len(valid_files)} (source={SOURCE})")
SPLITS, TOTAL = ctd.make_split(train_files, valid_files, mode=CFG.split_mode, limits={})
for name, files in SPLITS.items():
    print(
        f"  split {name}: {len(files)} images  {files[0].rsplit('/', 1)[-1]}..{files[-1].rsplit('/', 1)[-1]}"
    )
ctd.write_dataset_split_json(CFG, SPLITS)

### Stage B - cached window dataset (actual embedder)
For every image the registry payload `id_registry.encode_id_bits(id)` is embedded with
`src.watermark.embed.embed` at its native HR size, then the watermarked pixels are
downscaled to 256 and the **authoritative** per-bit windows (`bit_window_singular_values` -
identical mapping at train and inference) stored to `*.npz`. Labels are those exact
embedded bits. The cache is keyed by a config hash and reused on resume / across stages.

In [ ]:
# resume-safe: kernel restarts clear globals - re-import the pipeline module + config
if "ctd" not in globals():
    import importlib.util
    import sys
    if "PROJECT_ROOT" not in globals():
        PROJECT_ROOT = "/content/deep-watermarking"
    sys.path.insert(0, PROJECT_ROOT)
    if not sys.modules.get("ctd_mod"):
        spec = importlib.util.spec_from_file_location(
            "ctd_mod", f"{PROJECT_ROOT}/training/colab_train_decoder.py"
        )
        mod = importlib.util.module_from_spec(spec)
        sys.modules["ctd_mod"] = mod
        spec.loader.exec_module(mod)
    ctd = sys.modules["ctd_mod"]
if "CFG" not in globals():
    CFG = ctd.load_pipeline_config(PROJECT_ROOT, {}).resolved()

if "SPLITS" not in globals():
    _json = __import__('json')
    _Path = __import__('pathlib').Path
    _payload = _json.loads(_Path(CFG.out_dir, "dataset_split.json").read_text())
    SPLITS = {name: [_Path(p) for p in files] for name, files in _payload["all_files"].items()}
    print("SPLITS reloaded from dataset_split.json")

CACHE = ctd.generate_cache(CFG, SPLITS)
print("manifest hash:", CFG.config_hash())

### Stage C - train the windowed CNN (GPU, resumable)
`windowed_cnn_best.pt` saves whenever validation improves; `windowed_cnn_last.pt` every
epoch. An existing `best.pt` auto-resumes - re-running this cell continues training
instead of restarting. Early stopping (patience `CFG.patience`) guards against
over-fitting. History lands in `training_history.csv`.

In [ ]:
# resume-safe: kernel restarts clear globals - re-import the pipeline module + config
if "ctd" not in globals():
    import importlib.util
    import sys
    if "PROJECT_ROOT" not in globals():
        PROJECT_ROOT = "/content/deep-watermarking"
    sys.path.insert(0, PROJECT_ROOT)
    if not sys.modules.get("ctd_mod"):
        spec = importlib.util.spec_from_file_location(
            "ctd_mod", f"{PROJECT_ROOT}/training/colab_train_decoder.py"
        )
        mod = importlib.util.module_from_spec(spec)
        sys.modules["ctd_mod"] = mod
        spec.loader.exec_module(mod)
    ctd = sys.modules["ctd_mod"]
if "CFG" not in globals():
    CFG = ctd.load_pipeline_config(PROJECT_ROOT, {}).resolved()

if "CACHE" not in globals():
    raise RuntimeError("run Cell 9 first (CACHE must exist)")

TRAINED = ctd.train_model(CFG, CACHE)
print({k: v for k, v in TRAINED.items() if k not in ("model", "rows")})

### Stage D - honest test-set evaluation
- **bit-level**: accuracy / BER / confusion matrix / precision / recall / F1 on the 64
  embedded bits;
- **full-payload registry-ID recovery**: for each test image the predicted 64 bits are
  decoded through the real `id_registry.decode_id_bits` majority decoder and compared to
  the true ID -> exact-ID rate (plus registered-at-threshold rate);
- **false-positive control**: clean (never-watermarked) test pixels through the same
  decoder - a registered ID at confidence >= 0.78 there is a false positive;
- **robustness smoke**: PNG re-encode, JPEG q90 and mild blur re-embedding BER.

In [ ]:
# resume-safe: kernel restarts clear globals - re-import the pipeline module + config
if "ctd" not in globals():
    import importlib.util
    import sys
    if "PROJECT_ROOT" not in globals():
        PROJECT_ROOT = "/content/deep-watermarking"
    sys.path.insert(0, PROJECT_ROOT)
    if not sys.modules.get("ctd_mod"):
        spec = importlib.util.spec_from_file_location(
            "ctd_mod", f"{PROJECT_ROOT}/training/colab_train_decoder.py"
        )
        mod = importlib.util.module_from_spec(spec)
        sys.modules["ctd_mod"] = mod
        spec.loader.exec_module(mod)
    ctd = sys.modules["ctd_mod"]
if "CFG" not in globals():
    CFG = ctd.load_pipeline_config(PROJECT_ROOT, {}).resolved()

if "CACHE" not in globals() or "TRAINED" not in globals():
    raise RuntimeError("run Cells 9 and 11 first (CACHE/TRAINED must exist)")

EVAL = ctd.evaluate_model(CFG, CACHE, TRAINED)
ctd.write_metrics(CFG, TRAINED, EVAL)
print("bit-level:", EVAL["bit_level"]["bit_accuracy"], "| ber:", EVAL["bit_level"]["ber"])
print("exact ID:", EVAL["full_payload"]["exact_id"], "/", EVAL["full_payload"]["test_images"])
print(
    "registered @0.78:",
    EVAL["full_payload"]["registered_at_threshold"],
    "/",
    EVAL["full_payload"]["test_images"],
)
print("false positives on clean:", EVAL["false_positives"]["false_positive_registrations"])

### Stage E - package + bring back
Writes `windowed_cnn_model_package.zip` (best + last checkpoints, `normalization.npz`,
`decoder_config.json`, `training_config.json`, `dataset_split.json`, `metrics.json`,
`training_history.csv`) and copies the same into `models/experimental/windowed_cnn/`. The
download button saves the zip; drop it into the repo and ingest it as documented.

In [ ]:
# resume-safe: kernel restarts clear globals - re-import the pipeline module + config
if "ctd" not in globals():
    import importlib.util
    import sys
    if "PROJECT_ROOT" not in globals():
        PROJECT_ROOT = "/content/deep-watermarking"
    sys.path.insert(0, PROJECT_ROOT)
    if not sys.modules.get("ctd_mod"):
        spec = importlib.util.spec_from_file_location(
            "ctd_mod", f"{PROJECT_ROOT}/training/colab_train_decoder.py"
        )
        mod = importlib.util.module_from_spec(spec)
        sys.modules["ctd_mod"] = mod
        spec.loader.exec_module(mod)
    ctd = sys.modules["ctd_mod"]
if "CFG" not in globals():
    CFG = ctd.load_pipeline_config(PROJECT_ROOT, {}).resolved()

ZIP_PATH = ctd.package_artifacts(CFG)
ctd.load_checkpoints_into_repo(CFG)
print("zip:", ZIP_PATH)
import os

for name in ["windowed_cnn_model_package.zip", "metrics.json", "training_history.csv"]:
    p = os.path.join(CFG.out_dir, name)
    if os.path.isfile(p):
        from google.colab import files

        print("downloading", name)
        files.download(p)

### Local usage once the artifacts are downloaded
```
# bring the trained decoder into the project (from the repo root):
unzip -o windowed_cnn_model_package.zip -d models/experimental/windowed_cnn/

# run the app with the windowed decoder (production decoder stays the default):
DECODER_MODE=windowed_cnn .venv/bin/python scripts/run_app.py
```
See `docs/windowed_cnn.md` for the full integration notes and the honest evaluation.